# Phase 1 Validation - V2 BERT Classification

**Purpose**: Run V2 BERT classifier on validation sample using Google Colab GPU

**Script**: `scripts/03a_run_v2_classification.py`

**Environment**: Google Colab with GPU support

**Runtime**: ~5 minutes (148 papers, GPU) | ~30 minutes (CPU)

---

## Usage

1. **Runtime**: Select GPU runtime (Runtime → Change runtime type → GPU)
2. **Execute cells**: Run cells in order from top to bottom
3. **Monitor**: Watch progress in cell outputs
4. **Results**: Automatically saved back to Google Drive

---

In [ ]:
# =============================================================================
# CELL 1: MOUNT GOOGLE DRIVE
# =============================================================================
# IMPORTANT: This MUST be the first cell

from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted successfully")
print("📦 Validation files are now accessible")
print("🔗 Ready for V2 BERT classification")

In [ ]:
# =============================================================================
# CELL 2: CONFIGURATION
# =============================================================================
import os
from datetime import datetime

# =============================================================================
# MODE SELECTION
# =============================================================================
TEST_MODE = False  # Set to True for quick testing (~5 minutes with 15 papers)
                   # Set to False for full validation (~5 minutes with 148 papers)

# Session Management (without _test suffix - TEST_MODE just limits processing)
import random
import string
from datetime import datetime

SESSION_ID = f"{datetime.now().strftime('%Y-%m-%d')}-{''.join(random.choices(string.ascii_lowercase + string.digits, k=6))}"
TIMESTAMP = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

print(f"\n🔖 Session ID: {SESSION_ID}")
print(f"📅 Timestamp: {TIMESTAMP}")

# =============================================================================
# PATH CONFIGURATION
# =============================================================================
# Base paths
BASE_PATH = '/content/drive/MyDrive'
REPO_NAME = 'inventory_2022'  # CONFIGURABLE: Change if your repo has different name
REPO_PATH = os.path.join(BASE_PATH, REPO_NAME)

# Validate repository exists
if not os.path.exists(REPO_PATH):
    raise FileNotFoundError(
        f"❌ Repository not found at: {REPO_PATH}\n"
        f"   Please ensure:\n"
        f"   1. Google Drive is mounted\n"
        f"   2. Repository folder exists in Drive root\n"
        f"   3. Update REPO_NAME variable if folder has different name"
    )

VALIDATION_PATH = os.path.join(REPO_PATH, 'validation_spacy_v_BERT')

# Validate validation directory exists
if not os.path.exists(VALIDATION_PATH):
    raise FileNotFoundError(
        f"❌ Validation directory not found at: {VALIDATION_PATH}\n"
        f"   Please ensure validation_spacy_v_BERT folder exists in repository"
    )

print(f"✓ Repository found: {REPO_PATH}")
print(f"✓ Validation directory: {VALIDATION_PATH}")

# Change to validation directory
os.chdir(VALIDATION_PATH)

# Set TEST_MODE environment variable for script
os.environ['TEST_MODE'] = str(TEST_MODE)
os.environ['SESSION_ID'] = SESSION_ID  # Pass session ID to scripts

# Mode-specific configuration
if TEST_MODE:
    print("\n🧪 TEST MODE ENABLED")
    print("   ⏱️  Expected runtime: ~2-3 minutes")
    print("   📊 Dataset: 15 papers (test subset)")
else:
    print("\n🚀 PRODUCTION MODE ENABLED")
    print("   ⏱️  Expected runtime: ~5 minutes")
    print("   📊 Dataset: 148 papers (full validation sample)")

print(f"\n📁 Working directory: {os.getcwd()}")
print(f"📁 Repository path: {REPO_PATH}")

print("\n" + "="*60)
print("CONFIGURATION LOADED")
print("="*60)

In [ ]:
# =============================================================================
# CELL 3: INSTALL DEPENDENCIES
# =============================================================================

print("🔧 Installing packages...")
!pip install transformers datasets evaluate seqeval nltk -q

print("\n✅ Dependencies installed")

# Download NLTK data
import nltk
import ssl

print("Downloading NLTK data...")
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('punkt_tab', quiet=True)
print("✅ NLTK data downloaded")

print("\n✅ Environment setup complete")

In [ ]:
# =============================================================================
# CELL 4: GPU CHECK
# =============================================================================

import torch

print("🔍 GPU Environment Check:")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
    print("✅ GPU acceleration available")
    print("\n⚡ Classification will run ~5-10x faster on GPU")
else:
    print("⚠️ No GPU detected - inference will be slower")
    print("Consider enabling GPU runtime: Runtime > Change runtime type > GPU")
    print("\n⏱️  Expected runtime on CPU: ~30 minutes")

In [ ]:
# =============================================================================
# CELL 5: VERIFY INPUT FILES
# =============================================================================

import sys
from pathlib import Path

# Add project root to Python path
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

# Also add src directory for inventory_utils imports
src_path = str(Path(REPO_PATH) / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("🔍 Verifying input files...\n")

# Check validation sample
SESSION_ID = os.environ.get('SESSION_ID', '')
output_suffix = f"_{SESSION_ID}" if SESSION_ID else ("_test" if TEST_MODE else "")
sample_file = Path(f"results/validation/sample/validation_sample_with_abstracts{output_suffix}.csv")

if sample_file.exists():
    print(f"✓ Validation sample: {sample_file}")
    print(f"  Size: {sample_file.stat().st_size / 1024:.1f} KB")
else:
    print(f"❌ Validation sample not found: {sample_file}")
    print("\nPlease run scripts 01 and 02 first:")
    print("  1. TEST_MODE=True python scripts/01_select_validation_sample.py")
    print("  2. TEST_MODE=True python scripts/02_fetch_abstracts.py")
    raise FileNotFoundError(f"Required file not found: {sample_file}")

# Check V2 model
model_path = Path(f"{REPO_PATH}/out/original_model/article_classifier.pt")

if model_path.exists():
    print(f"\n✓ V2 Classification model: {model_path}")
    print(f"  Size: {model_path.stat().st_size / 1024 / 1024:.1f} MB")
else:
    print(f"\n❌ V2 model not found: {model_path}")
    print("\nPlease ensure V2 models are uploaded to Google Drive")
    raise FileNotFoundError(f"Required model not found: {model_path}")

# Check script exists
script_path = Path("scripts/03a_run_v2_classification.py")
if not script_path.exists():
    raise FileNotFoundError(f"Required script not found: {script_path}")
print(f"\n✓ Script verified: {script_path}")

print("\n✅ All required files verified")

In [ ]:
# =============================================================================
# CELL 6: RUN V2 BERT CLASSIFICATION
# =============================================================================

print("🚀 Running V2 BERT Classification...\n")
print("="*70)

# Run the classification script
%run scripts/03a_run_v2_classification.py

print("\n" + "="*70)
print("✅ V2 BERT Classification Complete")

In [ ]:
# =============================================================================
# CELL 7: DISPLAY RESULTS
# =============================================================================
import pandas as pd

# Re-establish context for standalone execution
TEST_MODE = os.environ.get('TEST_MODE', 'False').lower() == 'true'
SESSION_ID = os.environ.get('SESSION_ID', '')
output_suffix = f"_{SESSION_ID}" if SESSION_ID else ("_test" if TEST_MODE else "")

print("📊 Classification Results Summary\n")
print("="*70)

# Load results
results_file = Path(f"results/validation/classification/v2_classification_results{output_suffix}.csv")

if results_file.exists():
    df_results = pd.read_csv(results_file)
    
    print(f"\n📄 Results file: {results_file}")
    print(f"📦 File size: {results_file.stat().st_size / 1024:.1f} KB")
    
    print(f"\n📊 Classification Statistics:")
    print(f"   Total papers: {len(df_results)}")
    
    if 'predicted_label' in df_results.columns:
        bio_count = (df_results['predicted_label'] == 'bio-resource').sum()
        not_bio_count = (df_results['predicted_label'] == 'not-bio-resource').sum()
        
        print(f"   Predicted bio-resource: {bio_count} ({100*bio_count/len(df_results):.1f}%)")
        print(f"   Predicted not-bio-resource: {not_bio_count} ({100*not_bio_count/len(df_results):.1f}%)")
        
        # Show confidence distribution
        if 'confidence' in df_results.columns:
            print(f"\n📈 Confidence Scores:")
            print(f"   Mean: {df_results['confidence'].mean():.3f}")
            print(f"   Median: {df_results['confidence'].median():.3f}")
            print(f"   Min: {df_results['confidence'].min():.3f}")
            print(f"   Max: {df_results['confidence'].max():.3f}")
    
    # Show sample results
    print(f"\n📋 Sample Results (first 5):")
    display_cols = ['publication_id', 'predicted_label']
    if 'confidence' in df_results.columns:
        display_cols.append('confidence')
    
    print(df_results[display_cols].head())
    
    print(f"\n✅ Results successfully saved to Drive")
    print(f"   Location: {results_file}")
    
else:
    print(f"⚠️ Results file not found: {results_file}")
    print("Classification may have failed - check output above")

print("\n" + "="*70)

In [ ]:
# =============================================================================
# CELL 8: COMPLETION SUMMARY
# =============================================================================

from datetime import datetime
from pathlib import Path

# Re-establish context for standalone execution
TEST_MODE = os.environ.get('TEST_MODE', 'False').lower() == 'true'
SESSION_ID = os.environ.get('SESSION_ID', '')
output_suffix = f"_{SESSION_ID}" if SESSION_ID else ("_test" if TEST_MODE else "")
results_file = Path(f"results/validation/classification/v2_classification_results{output_suffix}.csv")

print("\n" + "🎉" * 35)
print("🎉 V2 BERT CLASSIFICATION COMPLETE")
print("🎉" * 35)

print(f"\n📊 Session Information:")
print(f"   Session ID: {SESSION_ID if SESSION_ID else 'N/A'}")
print(f"   Mode: {'TEST' if TEST_MODE else 'PRODUCTION'}")
print(f"   Completion Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print(f"\n📁 Output Files:")
if results_file.exists():
    print(f"   📄 {results_file}")
else:
    print(f"   ⚠️ Results file not created (check errors above)")

print(f"\n🎯 Next Steps:")
print(f"   1. Review results above")
print(f"   2. Run PyCaret classification: scripts/03b_run_pycaret_classification.py")
print(f"   3. Or continue with NER: validation_v2_ner.ipynb")

print(f"\n🏁 Notebook execution complete!")
print(f"📊 Results are automatically saved to Google Drive")